# agent

> The demo agent: a model writes a spec from the block manifests, and code checks it before anyone sees it.

`author(concept, ask)` asks a model for one demo on a concept. The model writes data only: a JSON spec. It never writes code.
Then the loop checks the spec, and gives the errors back to the model to fix:

1. Parse the reply as JSON.
2. `validate` the spec: types, binds, gates and the think rule.
3. `check` the spec in headless Chrome: it draws, it plays, the keys work, the contrast is good.
4. On errors, ask for a repair with only these errors. At most two repairs.
5. Give up quietly: return `None` and log the errors. The caller keeps its text lesson.

`ask(prompt, sp)` is any function that returns text, as in abhyasa. The tests use a fake one.

In [ ]:
#| default_exp agent

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| export
import json, re
from fastcore.utils import *
from rupaka.spec import manifests, fn_manifests, validate, gallery, SCHEMA

## The catalog

The prompt lists the blocks and fns from their manifests, so it never names a block that the runtime does not have.

In [ ]:
#| export
def _prop(k, p):
    bits = [p['type'], 'required' if p.get('req') else None, f'default {json.dumps(p["default"], ensure_ascii=False)}' if 'default' in p else None]
    return f'{k} ({", ".join(b for b in bits if b)})' + (f': {p["doc"]}' if p.get('doc') else '')

def catalog() -> str:
    "The blocks and fns, with their purpose and props, as text for the prompt."
    bl = [f'- {m.name}: {m.purpose}' + (f' Keys: {m.keys}.' if m.get('keys', 'none') != 'none' else '')
          + (' Can have "lock".' if m.get('lockable') else '') + (f' Only inside {m.parent}.' if m.get('parent') else '')
          + ''.join(f'\n    {_prop(k, p)}' for k, p in m.props.items()) for m in manifests().values()]
    fs = [f'- {m.name}: {m.purpose} Returns {m.returns}. Example: {json.dumps(m.example, ensure_ascii=False)}'
          + ''.join(f'\n    {_prop(k, p)}' for k, p in m.props.items()) for m in fn_manifests().values()]
    return 'BLOCKS (every block can have "after": a gate id)\n' + '\n'.join(bl) + '\n\nFNS (each derive entry has "fn" and "out")\n' + '\n'.join(fs)

The catalog names every block and every fn.

In [ ]:
cat = catalog()
for n in [*manifests(), *fn_manifests()]: assert f'- {n}: ' in cat, n
print(cat[:600])

## The system prompt

The system prompt gives the rules of a good demo, the shape of a spec, and the catalog.

In [ ]:
#| export
SHAPE = '''{"rupaka": 1, "id": "short-slug", "title": str, "ask": str,
 "state": {key: value},            # the inputs
 "derive": [{"fn": name, ..., "out": key}],   # computed in order after each change
 "blocks": [{"type": name, ...}],
 "tables": {name: [rows]}}         # data for rules and lookup'''

def demo_sp() -> str:
    "The system prompt of the demo agent."
    return f"""You design one interactive demo that makes the learner think. You write a JSON spec. You never write code.
Rules:
- One concept. At least one gate. The learner predicts before the demo shows any result.
- Every block that shows a value the gate's answer depends on has "after": the gate id. Inputs can show before the gate; results cannot.
- Controls that the learner should not move before the prediction have "lock": the gate id.
- A gate answer is a state path (for example "fused.0.id") that is one of its options, or {{"is": value}} for a fixed answer with "hides": the keys that would give it away.
- When a control can change the answer after the reveal, the gate has "context" that names the inputs, for example "For {{a}} + {{b}}:".
- After the reveal, the controls are live. The last caption asks one new question that the learner can answer only by using the controls.
- Wrong options in a gate each match one named misconception. Use "rules" with "alts" when the concept is a rule system.
- Text uses {{path}} for state values, and **bold** in captions. No HTML.
- Fit in 640 x 720 pixels. At most 8 blocks. Write text in ASD-STE100 Simplified Technical English, US grade 9: short sentences of at most 20 words, active voice, simple words.
- Facts must be correct. Use only facts that you are sure of.
The shape of a spec:
{SHAPE}
Use only these blocks and fns:
{catalog()}
Return JSON only."""

In [ ]:
assert 'You never write code' in demo_sp() and '- wheel: ' in demo_sp()

## The user prompt

The user prompt describes the concept, the first error of the learner (if there is one), and two gallery demos as examples.
`near` picks the gallery demos that share the most words with the concept. A long table in an example is cut to three rows.

In [ ]:
#| export
def _words(s): return set(re.findall(r'[a-zāīūṛṃḥṅñṭḍṇśṣ]{3,}', str(s).lower()))

def near(concept, k:int=2) -> list:
    "The `k` gallery demos that share the most words with `concept`."
    w = _words(' '.join(str(concept.get(f, '')) for f in ('name', 'statement', 'subject')))
    gs = sorted(gallery().values(), key=lambda s: -len(w & _words(f"{s.get('title')} {s.get('ask')} {s.get('concept')}")))
    return gs[:k]

def _brief(spec):
    "A copy of `spec` with each table cut to three rows."
    return {**spec, 'tables': {k: v[:3] for k, v in spec['tables'].items()}} if spec.get('tables') else spec

In [ ]:
#| export
def demo_prompt(concept, first_error:str='', examples=()) -> str:
    "The prompt that asks for a demo on `concept`, a dict with name, statement, subject and misconceptions."
    mis = '; '.join(concept.get('misconceptions') or []) or 'none known'
    ex = '\n\n'.join(json.dumps(_brief(e), ensure_ascii=False) for e in examples)
    return (f"Concept: {concept['name']} (subject: {concept.get('subject') or 'general'})\n"
            f"Statement: {concept.get('statement') or '(none given)'}\nMisconceptions seen: {mis}\n"
            + (f"The learner's first error: {first_error}\n" if first_error else '')
            + "\nDesign one demo that makes the learner predict, then see, then change the inputs."
            + (" Aim the gate at the first error and the misconceptions." if first_error or concept.get('misconceptions') else '')
            + (f"\n\nExamples of good demos (tables cut short):\n{ex}" if ex else '')
            + '\n\nReturn the JSON spec only.')

In [ ]:
c = dict(name='Reciprocal rank fusion', statement='Sum 1/(k + rank) over the lists.', subject='ml', misconceptions=['raw scores add'])
p = demo_prompt(c, 'You add the raw scores.', near(c))
assert 'first error: You add the raw scores.' in p and '"id": "reciprocal-rank-fusion"' in p

## The repair prompt

The repair prompt gives back the spec and only its errors.

In [ ]:
#| export
def repair_prompt(spec, errors) -> str:
    "The prompt that asks the model to fix only `errors` in `spec`."
    s = spec if isinstance(spec, str) else json.dumps(spec, ensure_ascii=False)
    return (f"This demo spec has errors.\n\nSpec:\n{s}\n\nErrors:\n" + '\n'.join(f'- {e}' for e in errors)
            + '\n\nFix only these errors. Keep the rest. Return the whole spec as JSON only.')

## Parse

`parse_json` finds the JSON object in a reply, also inside a fence or after some text.

In [ ]:
#| export
def parse_json(s:str) -> dict:
    "The first JSON object in `s`. Raise `ValueError` when there is none."
    s = s.strip()
    if (m := re.search(r'```(?:json)?\s*(.*?)```', s, re.S)): s = m.group(1).strip()
    try: return json.loads(s)
    except json.JSONDecodeError: pass
    i, j = s.find('{'), s.rfind('}')
    if 0 <= i < j:
        try: return json.loads(s[i:j + 1])
        except json.JSONDecodeError as e: raise ValueError(f'the reply is not valid JSON: {e}') from None
    raise ValueError('the reply has no JSON object')

In [ ]:
test_eq(parse_json('Here:\n```json\n{"a": 1}\n```'), {'a': 1})
test_fail(lambda: parse_json('no json'), contains='no JSON')

## author

`author` runs the loop. It stamps the spec with the schema, an id and the concept slug. `log`, when given, gets one entry for each failed try.

In [ ]:
#| export
def _slug(s): return re.sub(r'[^a-z0-9]+', '-', str(s).lower()).strip('-') or 'demo'

def author(concept, ask, first_error:str='', examples=None, check=True, repairs:int=2, log:list=None) -> dict|None:
    "A checked demo spec on `concept` from the model `ask`, or None after `repairs` failed repairs."
    c = concept if isinstance(concept, dict) else dict(name=str(concept))
    slug = c.get('slug') or _slug(c['name'])
    if check is True: from rupaka.check import check
    prompt, sp = demo_prompt(c, first_error, near(c) if examples is None else examples), demo_sp()
    log = [] if log is None else log
    for i in range(repairs + 1):
        try: reply = ask(prompt, sp)
        except Exception as e: log.append(dict(tries=i + 1, errors=[f'model: {e}'])); return None
        try: spec = parse_json(reply)
        except ValueError as e: spec, errs = None, [f'reply: {e}']
        else:
            spec = {**spec, 'rupaka': SCHEMA, 'id': spec.get('id') or slug, 'concept': slug}
            errs = list(validate(spec))
            if not errs and check: errs = list(check(spec).errors)
            if not errs: return spec
        log.append(dict(tries=i + 1, errors=errs))
        prompt = repair_prompt(spec or reply, errs)
    return None

### Tests with a fake model

`fake` replies from a list, one reply for each call, and keeps the prompts.

In [ ]:
from rupaka.spec import examples as block_examples

In [ ]:
def fake(*replies):
    calls = []
    def ask(p, sp=''): calls.append(p); return replies[min(len(calls), len(replies)) - 1]
    ask.calls = calls
    return ask
good = json.dumps(block_examples()['gate'])

A good spec passes on the first call. The spec gets the concept slug.

In [ ]:
a = fake(good)
s = author(dict(name='Sums', slug='sums'), a, check=False)
test_eq((s['concept'], len(a.calls)), ('sums', 1))

A spec with one error gets one repair. The repair prompt names the error.

In [ ]:
broken = json.loads(good); del broken['blocks'][1]['after']
a, log = fake(json.dumps(broken), good), []
s = author('Sums', a, check=False, log=log)
assert s is not None and len(a.calls) == 2
assert 'gives away the answer of gate "sum"' in a.calls[1] and 'Fix only these errors' in a.calls[1]
test_eq(log[0]['tries'], 1)

Three bad replies give up quietly: the result is `None`, and the log has three entries.

In [ ]:
log = []
test_eq(author('Sums', fake('not json', json.dumps(broken), '{}'), check=False, log=log), None)
test_eq(len(log), 3)

A model that fails gives up too, with no exception.

In [ ]:
def down(p, sp=''): raise ConnectionError('no network')
test_eq(author('Sums', down, check=False), None)

With a browser, `check` runs too, and a good spec still passes in one call.

In [ ]:
from rupaka.check import can_check

In [ ]:
if can_check(): assert author('Sums', fake(good)) is not None

## A live run

A live run uses a real model, so it is not part of the tests. With the `rishi` package and its model `claude/sonnet`, it looks like this:

```python
from rishi import Chat, resp_text
def ask(p, sp=''):
    chat = Chat('claude/sonnet', sp=sp)
    try: return resp_text(chat(p))
    finally: chat.close()
log = []
spec = author(dict(name='Guṇa sandhi', statement='a or ā before i, u or ṛ gives e, o or ar.', subject='sanskrit'), ask, log=log)
```

The first live run (26 October 2026, `claude/sonnet`, three concepts: guṇa sandhi, cosine similarity and binary search):

| concept | result | model calls |
|---|---|---|
| guṇa sandhi | passed on the first try | 1 |
| cosine similarity | passed after one repair: a point showed the answer before the gate, and the think rule caught it | 2 |
| binary search | passed on the first try | 1 |

So 3 of 3 passed, 2 of them on the first try. One spec put `{n}` in its plain-text `ask`. The check could not see it, so `validate` now refuses a template in a plain-text prop.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()